In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.base import clone

<div dir="rtl">

# کارگاه جلسه ۱۵ — Pipeline پیش‌پردازش

این دفترچه برای **تمرین** است و همراه جزوه جلسه ۱۵ استفاده می‌شود. نوشتن این کدها از حفظ در امتحان پرسیده نمی‌شود؛ هدف این است که هر گام را اجرا کنید، خروجی را ببینید و با مفاهیم جزوه ارتباط دهید.

داده: ۳۰ دانشجوی ساختگی با یک ویژگی کیفی (رشته) و هشت ویژگی عددی. پنج ویژگی تحصیلی (مطالعه، حضور، تمرین، میان‌ترم، پایان‌ترم) با هم و سه ویژگی مشغله (کار، رفت‌وآمد، خواب) با هم همبسته‌اند.
</div>

<div dir="rtl">

## ۱. داده خام

مشکلات داده خام: دو سطر تکراری (S07 و S19)، برچسب‌های ناسازگار (cs، math و Stats با فاصله اضافه)، ستون رفت‌وآمد متنی با علامت ? برای دو دانشجو، سه مقدار ناممکن (حضور ۹۲۰، میان‌ترم ۲۵ از ۲۰، خواب −1) و چند خانه گمشده.
</div>

In [2]:
raw = pd.DataFrame({
    "student": ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S07", "S08", "S09", "S10",
                "S11", "S12", "S13", "S14", "S15", "S16", "S17", "S18", "S19", "S19", "S20",
                "S21", "S22", "S23", "S24", "S25", "S26", "S27", "S28", "S29", "S30"],
    "major": ["Math", "Math", "CS", "Stats", "Stats ", "Stats", "Math", "Math", "CS", "Stats", "Math",
              np.nan, "Math", "cs", "Stats", "Stats", "CS", "CS", "CS", "CS", "CS", "Math",
              "CS", "math", "Math", np.nan, "Stats", "Stats", "CS", "Math", "CS", "CS"],
    "study_hours": [9, 10, np.nan, 10, 8, 5, 13, 13, 3, 10, 11, 5, 15, 5, 4, 6,
                    9, 8, 1, 9, 9, 10, 4, 6, 12, 9, 8, 15, 7, 9, 4, 10],
    "attendance": [72, 86, 78, 87, 77, 64, 95, 95, 57, 86, 91, 63, 95, 64, 66, 66,
                   920, 71, 49, 76, 76, 87, 57, 57, 90, 76, 80, 100, 61, 69, 62, 92],
    "assignment": [14, 15, 13, 16, 12, 12, 18, 18, 13, 16, 17, 13, 18, 10, 11, 11,
                   15, 13, 8, 15, 15, 15, 9, 11, 18, 14, 15, 20, 12, 13, 11, 16],
    "midterm": [14, 16, 13, 15, 13, 10, 18, 18, 12, 13, 14, 12, 19, 10, 9, 11,
                13, 12, 7, 14, 14, 25, 8, 9, 18, 12, 12, 20, 11, 12, 9, 16],
    "final": [14, 15, 13, 18, 12, 10, 17, 17, 10, 15, 15, 11, 18, 10, np.nan, 11,
              14, 13, 7, 13, 13, 15, 8, 12, 17, 14, 13, 20, 11, 12, 9, 15],
    "job_hours": [22, 8, 12, 6, 14, 12, 16, 16, 8, 6, 0, 10, 0, 12, 12, 12,
                  18, 4, 16, 8, 8, 2, 4, 4, 8, 18, np.nan, 26, 0, 26, 16, 8],
    "commute_min": ["70", "35", "35", "25", "40", "35", "55", "55", "40", "?", "20", "45", "20", "50", "30", "45",
                    "55", "20", "50", "40", "40", "25", "25", "20", "25", "45", "60", "65", "?", "70", "50", "30"],
    "sleep_hours": [5.5, 6.5, 6.5, 7.5, 6.5, 7.0, 6.5, 6.5, 7.5, 8.5, 7.5, 7.0, 8.0, 6.5, 7.0, 6.0,
                    6.5, 8.0, 6.5, 7.0, 7.0, 7.5, 7.5, 8.0, 7.5, 6.0, 6.5, 5.5, 8.5, -1, 6.0, 7.0],
})
print(raw.shape, raw.duplicated().sum())
print(raw["major"].unique().tolist())
print(raw.dtypes)

(32, 10) 2
['Math', 'CS', 'Stats', 'Stats ', nan, 'cs', 'math']
student            str
major              str
study_hours    float64
attendance       int64
assignment       int64
midterm          int64
final          float64
job_hours      float64
commute_min        str
sleep_hours    float64
dtype: object


<div dir="rtl">

## ۲. گام‌های قاعده‌محور (پیش از تقسیم)
</div>

In [3]:
df = raw.drop_duplicates().set_index("student")
df["major"] = df["major"].str.strip().replace({"cs": "CS", "math": "Math"})
df["commute_min"] = pd.to_numeric(df["commute_min"], errors="coerce")   # "?" -> NaN
df.loc[~df["attendance"].between(0, 100), "attendance"] = np.nan
df.loc[~df["midterm"].between(0, 20), "midterm"] = np.nan
df.loc[~df["sleep_hours"].between(0, 24), "sleep_hours"] = np.nan
print(len(df))
print(df["major"].value_counts(dropna=False).to_dict())
print(df.isna().sum().to_dict())

30
{'CS': 11, 'Math': 9, 'Stats': 8, nan: 2}
{'major': 2, 'study_hours': 1, 'attendance': 1, 'assignment': 0, 'midterm': 1, 'final': 1, 'job_hours': 1, 'commute_min': 2, 'sleep_hours': 1}


<div dir="rtl">

## ۳. تقسیم آموزش و آزمون
</div>

In [4]:
X_train, X_test = train_test_split(df, test_size=0.3, random_state=7)
print(len(X_train), len(X_test))
print(X_train.isna().sum()[X_train.isna().sum() > 0].to_dict())
print(X_test.isna().sum()[X_test.isna().sum() > 0].to_dict())

21 9
{'major': 2, 'attendance': 1, 'midterm': 1, 'final': 1, 'commute_min': 2}
{'study_hours': 1, 'job_hours': 1, 'sleep_hours': 1}


<div dir="rtl">

## ۴. ساخت Pipeline و خواندن پارامترهای یادگرفته‌شده
</div>

In [5]:
num_cols = ["study_hours", "attendance", "assignment", "midterm",
            "final", "job_hours", "commute_min", "sleep_hours"]
cat_cols = ["major"]

num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])

Z_train = preprocess.fit_transform(X_train)
Z_test = preprocess.transform(X_test)
print(Z_train.shape, Z_test.shape)
print(preprocess.get_feature_names_out())

(21, 11) (9, 11)
['num__study_hours' 'num__attendance' 'num__assignment' 'num__midterm'
 'num__final' 'num__job_hours' 'num__commute_min' 'num__sleep_hours'
 'cat__major_CS' 'cat__major_Math' 'cat__major_Stats']


In [6]:
num_part = preprocess.named_transformers_["num"]
print("medians:", num_part["impute"].statistics_)
print("means:  ", num_part["scale"].mean_.round(2))
print("stds:   ", num_part["scale"].scale_.round(2))
print("mode:   ", preprocess.named_transformers_["cat"]["impute"].statistics_)

medians: [ 8.  71.5 13.  12.  13.5 10.  40.   7. ]
means:   [ 7.95 74.17 13.71 12.71 13.31 10.76 40.    6.98]
stds:    [ 3.15 13.34  2.83  3.13  3.03  6.66 14.06  0.88]
mode:    ['CS']


<div dir="rtl">

**پرسش:** میانگین ستون‌های استانداردشده را در آموزش و آزمون مقایسه کنید. چرا در آزمون صفر نیست؟
</div>

In [7]:
print(Z_train[:, :8].mean(axis=0).round(3))
print(Z_test[:, :8].mean(axis=0).round(3))

[-0. -0.  0.  0. -0. -0.  0. -0.]
[ 0.156  0.171  0.101  0.091 -0.139 -0.081  0.04  -0.036]


<div dir="rtl">

## ۵. افزودن PCA به شاخه عددی

روش PCA فقط به شاخه عددی و پس از استانداردسازی اضافه می‌شود.
</div>

In [8]:
num_pca = Pipeline([("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                    ("pca", PCA(n_components=0.90))])
prep_pca = ColumnTransformer([("num", num_pca, num_cols),
                              ("cat", clone(cat_pipe), cat_cols)])
Zp_train = prep_pca.fit_transform(X_train)
pca = prep_pca.named_transformers_["num"]["pca"]
print(Zp_train.shape, prep_pca.get_feature_names_out())
print(pca.n_components_, pca.explained_variance_.round(3), pca.explained_variance_ratio_.round(3))
print(pd.DataFrame(pca.components_.T, index=num_cols, columns=["PC1", "PC2"]).round(2))

(21, 5) ['num__pca0' 'num__pca1' 'cat__major_CS' 'cat__major_Math'
 'cat__major_Stats']
2 [5.212 2.447] [0.621 0.291]


              PC1   PC2
study_hours  0.42  0.16
attendance   0.41  0.19
assignment   0.42  0.17
midterm      0.42  0.09
final        0.41  0.20
job_hours    0.25 -0.52
commute_min  0.19 -0.53
sleep_hours -0.18  0.56


<div dir="rtl">

**مقایسه:** اگر PCA بعد از کل پیش‌پردازش (روی ۱۱ ستون، شامل One-Hot) قرار گیرد، چند مؤلفه نگه داشته می‌شود؟ واریانس ستون‌های One-Hot را با واریانس ستون‌های استانداردشده مقایسه کنید.
</div>

In [9]:
wrong = Pipeline([("prep", clone(preprocess)), ("pca", PCA(n_components=0.90))]).fit(X_train)
print(wrong["pca"].n_components_, wrong["pca"].explained_variance_ratio_.round(3))
print(Z_train.var(axis=0).round(3))

3 [0.584 0.272 0.047]
[1.    1.    1.    1.    1.    1.    1.    1.    0.249 0.181 0.204]


<div dir="rtl">

## ۶. گام سفارشی با FunctionTransformer
</div>

In [10]:
def add_exam_avg(d):
    """Feature engineering step: average of midterm and final."""
    d = d.copy()
    d["exam_avg"] = (d["midterm"] + d["final"]) / 2
    return d

prep_fe = Pipeline([
    ("fe", FunctionTransformer(add_exam_avg)),
    ("prep", ColumnTransformer([("num", clone(num_pipe), num_cols + ["exam_avg"]),
                                ("cat", clone(cat_pipe), cat_cols)])),
])
print(prep_fe.fit_transform(X_train).shape)
print(prep_fe["prep"].named_transformers_["num"]["impute"].statistics_[-1])

(21, 12)
12.5


<div dir="rtl">

## ۷. تغییر پارامتر با set_params و خروجی DataFrame با set_output

نام پارامتر: نام گام‌ها از بیرون به درون با دو زیرخط، مثلاً `num__impute__strategy`.
</div>

In [11]:
prep_mean = clone(preprocess).set_params(num__impute__strategy="mean").fit(X_train)
print(prep_mean.named_transformers_["num"]["impute"].statistics_.round(2))

prep_df = clone(preprocess).set_output(transform="pandas")
Z_df = prep_df.fit_transform(X_train)
print(Z_df.iloc[:3, [0, 4]].round(3))
print(Z_df.iloc[:3, 8:])

[ 7.95 74.3  13.71 12.75 13.3  10.76 40.    6.98]
         num__study_hours  num__final
student                              
S30                 0.649       0.559
S13                -0.936      -1.094
S14                -1.253       0.063
         cat__major_CS  cat__major_Math  cat__major_Stats
student                                                  
S30                1.0              0.0               0.0
S13                1.0              0.0               0.0
S14                0.0              0.0               1.0


<div dir="rtl">

## ۸. دانشجوی جدید با رشته ناشناخته
</div>

In [12]:
new = pd.DataFrame([{"major": "Physics", "study_hours": np.nan, "attendance": 88,
                     "assignment": 16, "midterm": 15, "final": 16, "job_hours": 5,
                     "commute_min": 25, "sleep_hours": 7.5}])
print(preprocess.transform(new).round(3))

[[ 0.015  1.037  0.807  0.729  0.889 -0.865 -1.067  0.596  0.     0.
   0.   ]]


<div dir="rtl">

## ۹. انتخاب خودکار ستون‌ها بر اساس نوع

در داده خام، ستون commute_min متنی است و انتخابگر نوع آن را عددی نمی‌شناسد؛ پس از تبدیل نوع در گام پاک‌سازی، هر هشت ستون عددی انتخاب می‌شوند.
</div>

In [13]:
selector = make_column_selector(dtype_include="number")
print(selector(raw))
print(selector(df))

['study_hours', 'attendance', 'assignment', 'midterm', 'final', 'job_hours', 'sleep_hours']
['study_hours', 'attendance', 'assignment', 'midterm', 'final', 'job_hours', 'commute_min', 'sleep_hours']


<div dir="rtl">

## تمرین: داده خودروها (mpg)

مجموعه‌داده خودروها (۳۹۸ نمونه) را بارگذاری کنید. ستون mpg متغیر هدف جلسات آینده است و فعلاً کنار گذاشته می‌شود؛ ستون name شناسه است و حذف می‌شود. ستون origin (مبدأ خودرو) برای تقسیم طبقه‌بندی‌شده استفاده می‌شود.

1. تعداد مقادیر گمشده هر ستون را گزارش کنید. سهم هر مبدأ را در کل داده، آموزش و آزمون مقایسه کنید. همین تقسیم را بدون `stratify` تکرار کنید؛ سهم مبدأها در آزمون چه تغییری می‌کند؟
2. هر دو روش تقسیم را برای ۲۰۰ مقدار مختلف `random_state` تکرار کنید و کمترین، بیشترین و انحراف معیار سهم خودروهای اروپایی در آزمون را گزارش کنید.
3. یک ColumnTransformer بسازید: شش ستون عددی (جایگذاری میانه و استانداردسازی) و ستون origin (جایگذاری مد و کدگذاری One-Hot). آن را فقط روی داده آموزش `fit` کنید و ابعاد خروجی، نام ستون‌ها و میانه‌های یادگرفته‌شده را گزارش کنید.
4. میانگین ستون‌های عددی استانداردشده را در آموزش و آزمون مقایسه کنید. چرا در آزمون دقیقاً صفر نیست؟
5. با `FunctionTransformer` ویژگی «نسبت قدرت به وزن» (اسب بخار تقسیم بر وزن، ضرب در ۱۰۰۰) بسازید و به‌عنوان گام اول Pipeline اضافه کنید. ابعاد خروجی و میانه یادگرفته‌شده این ویژگی چیست؟
6. گام `PCA(n_components=0.90)` را به **شاخه عددی** تمرین ۳ اضافه کنید. چند مؤلفه نگه داشته می‌شود و خروجی چند ستون دارد؟ سپس با `set_params` راهبرد جایگذاری عددی را به میانگین تغییر دهید و مقدار جایگذاری horsepower را گزارش کنید.
7. شیء ColumnTransformer تمرین ۳ را روی یک خودروی جدید با اسب بخار گمشده و مبدأ ناشناخته korea اعمال کنید.
</div>

In [14]:
mpg = sns.load_dataset("mpg")
X = mpg.drop(columns=["mpg", "name"])
origin = mpg["origin"]
Xc_train, Xc_test = train_test_split(X, test_size=0.2, random_state=42, stratify=origin)
# your code here

<div dir="rtl">

### پاسخ‌های کلیدی تمرین

- فقط ستون horsepower **۶** مقدار گمشده دارد. داده آموزش ۳۱۸ و داده آزمون ۸۰ نمونه دارد. سهم مبدأها در کل داده و آموزش: usa حدود ۰.۶۲۶، japan حدود ۰.۱۹۸ و europe حدود ۰.۱۷۶؛ در آزمون طبقه‌بندی‌شده ۰.۶۲۵، ۰.۲۰۰ و ۰.۱۷۵. بدون `stratify` سهم‌ها در آزمون ۰.۶۶۲، ۰.۱۶۲ و ۰.۱۷۵ است.
- سهم خودروهای اروپایی در آزمون با تقسیم تصادفی ساده بین ۰.۰۷۵ و ۰.۳۰۰ نوسان می‌کند (انحراف معیار ۰.۰۳۹۲)؛ با تقسیم طبقه‌بندی‌شده همیشه ۰.۱۷۵ است.
- خروجی: ۳۱۸ در ۹ برای آموزش و ۸۰ در ۹ برای آزمون (۶ ستون عددی و ۳ ستون origin). میانه‌های یادگرفته‌شده: cylinders برابر ۴، displacement برابر ۱۴۶، horsepower برابر ۹۵، weight برابر ۲۸۳۱.۵، acceleration برابر ۱۵.۵ و model_year برابر ۷۶.
- میانگین ستون‌های عددی در آموزش صفر و در آزمون حدود −0.121، −0.132، −0.166، −0.112، 0.077 و 0.196 است؛ چون پارامترهای استانداردسازی از داده آموزش آمده‌اند.
- با ویژگی جدید، خروجی آموزش ۳۱۸ در ۱۰ است و میانه نسبت قدرت به وزن حدود ۳۴.۲۹ است.
- روش PCA نود درصدی در شاخه عددی **۳ مؤلفه** نگه می‌دارد (سهم‌ها حدود ۰.۷۱۴، ۰.۱۴۳ و ۰.۱۰۶؛ تجمعی ۰.۹۶۳) و خروجی ۶ ستون دارد (۳ مؤلفه و ۳ ستون origin). با راهبرد میانگین، مقدار جایگذاری horsepower حدود ۱۰۵.۷۹ است (در برابر میانه ۹۵).
- برای خودروی جدید، اسب بخار با میانه آموزش (۹۵) پر می‌شود و هر سه ستون origin صفر می‌شوند؛ خطایی رخ نمی‌دهد.
</div>